In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *
from eval.eval import *

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
############################################################

In [ ]:
### SETTINGS ###
config = load_config()

OmegaConf.set_struct(config, False)
config.pc_dim = 100
adata = process_data(pc_dim=config.pc_dim, data="cite")
timepoints = sorted(adata.obs['timepoint'].unique().tolist())
tree = adata.uns['tree']

config.num_classes = adata.obs['cell_type'].nunique()

project = "cite"

/home/azweig/projects/finfm/utils/lineage.py:75: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns['stoi'] = stoi


In [7]:
# # CFM
# config.metric = "cfm"
# config.no_learning = True

In [8]:
# MFM
config.metric = "mfm"
config.mfm.use_euclidean_ot = True
config.mfm.K = 150
config.mfm.kappa = 1.5
config.mfm.epsilon = 1e-1

# config.finsler.use = False
# config.finsler.lamb = 3.0
# config.dummy_kl_weight = 0.1

config.classifier_max_epochs = 2
config.metric_max_epochs = 3000
config.embed_max_epochs = 3000
config.flow_max_epochs = 2


In [9]:
adata.obs['donor'].unique()

array([32606])

In [10]:
t_holdout_index = 1
# t_holdout_index = 2

t0 = timepoints[t_holdout_index-1]
t = timepoints[t_holdout_index]
t1 = timepoints[t_holdout_index+1]
adata = adata[adata.obs['timepoint'].isin([t0, t, t1])]
adata_train = adata[adata.obs['timepoint'].isin([t0, t1])]

In [11]:
############################################################

In [12]:
singleton_dataloader = build_singleton_dataloader(config, adata_train)
paired_dataloader = build_paired_dataloader(config, adata_train)

In [13]:

classifier_model, metric_model, embed_model, flow_model = run_full_model(config=config,
                                                                         project=project,
                                                                         singleton_dataloader=singleton_dataloader,
                                                                         paired_dataloader=paired_dataloader,
                                                                         timepoints=timepoints,
                                                                         tree=tree)

DEBUG: trying dumb first loss as scaling for embed
Running phase classifier:.......


wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params | Mode 
----------------------------------------------------

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2048x100 and 5x20)

In [ ]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [ ]:
print(predict(embed_model, adata, t0, t, t1, num_traj=6000, library="pot"))